# Lab 1 – Del A: Klassificering
**Kurs:** Tillämpad maskininlärning 2026  
**Dataset:** Titanic (via seaborn)  
**Mål:** Predicera om en passagerare överlevde (survived = 1) eller inte (survived = 0)  
**Process:** CRISP-DM

## 1. Imports

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 2. Problem- och dataförståelse

### Problemformulering
Vi ska predicera om en passagerare på Titanic överlevde eller inte – ett **binärt klassificeringsproblem**.  
Målvariabeln är `survived` (0 = dog, 1 = överlevde).

### Vad representerar en rad?
Varje rad är en unik passagerare med attribut som ålder, klass, kön, biljettspris m.m.

### Attribut
| Attribut | Typ | Beskrivning |
|---|---|---|
| pclass | Numerisk (ordinal) | Biljettklass (1, 2, 3) |
| sex | Kategorisk | Kön |
| age | Numerisk | Ålder i år |
| sibsp | Numerisk | Antal syskon/make ombord |
| parch | Numerisk | Antal föräldrar/barn ombord |
| fare | Numerisk | Biljettspris |
| embarked | Kategorisk | Ombordstigning (C, Q, S) |

In [ ]:
# Ladda Titanic-datasetet via seaborn
df = sns.load_dataset('titanic')
print('Shape:', df.shape)
df.head()

In [ ]:
# Översikt över datatyper och saknade värden
print(df.dtypes)
print('\nSaknade värden per kolumn:')
print(df.isnull().sum())

In [ ]:
df.describe()

In [ ]:
# Klass-balans
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['survived'].value_counts()
axes[0].bar(['Dog (0)', 'Överlevde (1)'], counts.values, color=['#d9534f', '#5cb85c'])
axes[0].set_title('Klassfördelning – survived')
axes[0].set_ylabel('Antal passagerare')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, f'{v} ({v/len(df)*100:.1f}%)', ha='center')

# Överlevnad per kön
sns.countplot(data=df, x='sex', hue='survived', ax=axes[1],
              palette={0: '#d9534f', 1: '#5cb85c'})
axes[1].set_title('Överlevnad per kön')

plt.tight_layout()
plt.show()

print(f'\nKlass-balans: {counts[0]} dog ({counts[0]/len(df)*100:.1f}%) '
      f'vs {counts[1]} överlevde ({counts[1]/len(df)*100:.1f}%)')
print('Lätt obalans – vi använder stratifierad uppdelning och F1-score som primärt mått.')

In [ ]:
# Åldersfördelning per klass och överlevnadsstatus
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['age'].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Åldersfördelning')
axes[0].set_xlabel('Ålder')
axes[0].set_ylabel('Antal')

sns.boxplot(data=df, x='pclass', y='fare', hue='survived',
            palette={0: '#d9534f', 1: '#5cb85c'}, ax=axes[1])
axes[1].set_title('Biljettspris per klass och överlevnad')
axes[1].set_ylim(0, 300)

plt.tight_layout()
plt.show()

## 3. Dataförberedelse

**Val och motivering:**
- Behåller: `pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`, `embarked` — direkt relevanta features
- Tar bort: `deck` (77% saknas), redundanta kolumner som `who`, `adult_male`, `alive`, `embark_town`, `class`
- `age`: imputeras med medianen (robust mot outliers)
- `embarked`: imputeras med mode (2 saknade värden)
- `sex` och `embarked` kodas numeriskt
- Data skalas med StandardScaler för kNN (kNN baseras på avstånd och påverkas av skala)
- Träning/test: 80/20 med stratifiering för att bevara klassbalansen

In [ ]:
# Välj relevanta kolumner
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target = 'survived'

df_clean = df[features + [target]].copy()

# Imputera saknade värden
df_clean['age'].fillna(df_clean['age'].median(), inplace=True)
df_clean['embarked'].fillna(df_clean['embarked'].mode()[0], inplace=True)

print('Saknade värden efter imputering:')
print(df_clean.isnull().sum())

In [ ]:
# Koda kategoriska variabler
le_sex = LabelEncoder()
df_clean['sex'] = le_sex.fit_transform(df_clean['sex'])  # female=0, male=1

# One-hot encoding för embarked
df_clean = pd.get_dummies(df_clean, columns=['embarked'], drop_first=True)

print('Kolumner efter encoding:')
print(df_clean.columns.tolist())
print('Shape:', df_clean.shape)

In [ ]:
# Korrelationsheatmap
plt.figure(figsize=(8, 6))
corr = df_clean.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Korrelationsmatris')
plt.tight_layout()
plt.show()

In [ ]:
# Dela upp i features och mål
X = df_clean.drop(target, axis=1)
y = df_clean[target]

# Train/test split – stratifierad för att bevara klassbalansen
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Träning: {X_train.shape[0]} rader, Test: {X_test.shape[0]} rader')
print(f'Klassbalans träning: {y_train.value_counts().to_dict()}')
print(f'Klassbalans test:    {y_test.value_counts().to_dict()}')

In [ ]:
# Skala data – skalas separat för att undvika data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform på träningsdata
X_test_scaled  = scaler.transform(X_test)        # bara transform på testdata

print('Skalning klar. Mean (train):', X_train_scaled.mean(axis=0).round(4))

## 4. Modellering

Vi tränar tre typer av modeller med **två konfigurationer** vardera:

| Modell | Config 1 | Config 2 |
|---|---|---|
| kNN | k=3 (lokal, känslig) | k=11 (jämnare gräns) |
| Beslutsträd | max_depth=3 (tolkning) | max_depth=10 (flexibelt) |
| Random Forest | 50 träd | 200 träd |

**kNN** kräver skalad data (avståndsmått).  
**Beslutsträd och Random Forest** kräver inte skalning men vi testar båda för jämförbarhet.

In [ ]:
# Definiera modeller – kNN använder skalad data, träd använder oskalad
models = {
    'kNN k=3':           (KNeighborsClassifier(n_neighbors=3),  True),
    'kNN k=11':          (KNeighborsClassifier(n_neighbors=11), True),
    'Tree depth=3':      (DecisionTreeClassifier(max_depth=3,  random_state=RANDOM_STATE), False),
    'Tree depth=10':     (DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE), False),
    'RF 50 trees':       (RandomForestClassifier(n_estimators=50,  random_state=RANDOM_STATE), False),
    'RF 200 trees':      (RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE), False),
}

results = []

for name, (model, use_scaled) in models.items():
    Xtr = X_train_scaled if use_scaled else X_train
    Xte = X_test_scaled  if use_scaled else X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)

    results.append({
        'Modell':    name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
    })

results_df = pd.DataFrame(results).set_index('Modell')
print(results_df.round(3))

## 5. Utvärdering

In [ ]:
# Stapeldiagram – jämförelse av alla modeller
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(results_df))
width = 0.2
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i*width, results_df[metric], width, label=metric, color=color)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df.index, rotation=20, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Poäng')
ax.set_title('Modellprestanda – klassificering (Titanic)')
ax.legend()
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices för alla modeller
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()

for ax, (name, (model, use_scaled)) in zip(axes, models.items()):
    Xte = X_test_scaled if use_scaled else X_test
    y_pred = model.predict(Xte)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Dog', 'Överlevde'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)

plt.suptitle('Confusion matrices', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Detaljerad rapport för bästa modellen (Random Forest 200 träd)
best_model, best_scaled = models['RF 200 trees']
y_pred_best = best_model.predict(X_test)
print('=== Random Forest 200 träd – detaljerad rapport ===')
print(classification_report(y_test, y_pred_best, target_names=['Dog', 'Överlevde']))

In [ ]:
# Visualisera beslutsträd (depth=3) för tolkbarhet
tree_model = models['Tree depth=3'][0]
plt.figure(figsize=(18, 6))
plot_tree(
    tree_model,
    feature_names=X.columns.tolist(),
    class_names=['Dog', 'Överlevde'],
    filled=True, rounded=True, fontsize=9
)
plt.title('Beslutsträd (max_depth=3)')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance från Random Forest
rf_model = models['RF 200 trees'][0]
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=True)

plt.figure(figsize=(7, 5))
importances_sorted.plot(kind='barh', color='steelblue')
plt.title('Feature Importance – Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Korsvalidering (5-fold) för att bekräfta robusthet
print('5-fold korsvalidering (F1-score):')
for name, (model, use_scaled) in models.items():
    Xtr = X_train_scaled if use_scaled else np.array(X_train)
    scores = cross_val_score(model, Xtr, y_train, cv=5, scoring='f1')
    print(f'  {name:<20} mean={scores.mean():.3f}  std={scores.std():.3f}')

## 6. Slutsats och reflektion

### Resultatsammanfattning
- **Random Forest (200 träd)** gav bäst prestanda på testdatan med högst F1-score.
- **Beslutsträd med depth=10** visade tecken på **överanpassning** – hög träningsnoggrannhet men lägre F1 på test.
- **Beslutsträd med depth=3** är lättolkat och ger en rimlig grund, men sämre prestanda.
- **kNN med k=3** var känsligare för brus; k=11 gav jämnare resultat tack vare fler grannar.

### Påverkan av förbehandling
- Skalning var avgörande för kNN – utan skalning dominerar `fare` (stort värdeintervall) på bekostnad av andra features.
- Imputation av `age` med median var ett rimligt val (19% saknat).

### Varför fungerar Random Forest bäst?
Ensemblemetoder kombinerar många svaga modeller (träd) och minskar **varians** genom bagging. Random Forest introducerar också slumpmässighet i feature-urvalet vid varje split, vilket gör modellen mer robust mot outliers och kollinearitet.

### Viktigaste features (enligt Random Forest)
- `sex` och `fare` var de mest prediktiva – kön och biljettklass/pris korrelerar starkt med överlevnad.
- `age` var också viktig (barn räddades oftare).